# Фазовые метрики бокового сигнала

Ноутбук описывает фазу начала отклонения и фазу экстремума подписанных
ансамблей `33.03` на нормированной шкале интервала между принятыми
R-зубцами. Он не переводит фазу в секунды и не называет эти точки
открытием клапана, механической систолой или пиком кровенаполнения без
независимого временного референса.

## Определения

Фаза экстремума — положение максимального по модулю отклонения медианного
ансамбля на сетке `phase_rr` после R. Фаза onset определяется как первое
пересечение заданной доли (5, 10 или 20 %) от модуля того же знакового
экстремума, начиная с начала принятого окна ансамбля `phase_rr = 0`.
Следовательно, onset зависит от границы окна, медианного центрирования циклов,
фазовой сетки и выбранного порога. Это описательная метрика формы сигнала, а
не установленное начало механического события.

Фазовая шкала задаётся принятой парой соседних R-зубцов каждого цикла; она
не является абсолютной временной шкалой и не требует оценки групповой
задержки.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path

import numpy as np

REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def phase_metrics(phase_rr, waveform, thresholds=(0.05, 0.10, 0.20)):
    phase_rr = np.asarray(phase_rr, dtype=float)
    waveform = np.asarray(waveform, dtype=float)
    if len(phase_rr) < 2 or len(phase_rr) != len(waveform):
        raise ValueError("Нужен ансамбль с фазовой сеткой phase_rr")
    if not np.all(np.isfinite(phase_rr)) or not np.all(np.diff(phase_rr) > 0):
        raise ValueError("Фазовая сетка должна строго возрастать")
    if phase_rr[0] < 0.0 or phase_rr[-1] >= 1.0:
        raise ValueError("Ожидается сетка 0 <= phase_rr < 1")
    extremum_index = int(np.argmax(np.abs(waveform)))
    extremum_value = float(waveform[extremum_index])
    if extremum_value == 0 or not np.isfinite(extremum_value):
        raise RuntimeError("Нулевой ансамбль не имеет определимой фазы экстремума")
    onset = {}
    sign = np.sign(extremum_value)
    for fraction in thresholds:
        target = abs(extremum_value) * float(fraction)
        candidates = np.flatnonzero(
            (sign * waveform >= target)
            & (np.arange(len(waveform)) <= extremum_index)
        )
        onset[str(fraction)] = float(phase_rr[candidates[0]]) if len(candidates) else None
    available = [value for value in onset.values() if value is not None]
    return {
        "extremum_phase_rr": float(phase_rr[extremum_index]),
        "extremum_value_native": extremum_value,
        "onset_phase_rr_by_fraction": onset,
        "onset_method_spread_phase_rr": (
            float(max(available) - min(available)) if len(available) >= 2 else None
        ),
    }

In [ ]:
phase_test = np.arange(0.0, 1.0, 0.005)
wave_test = -0.020 * np.exp(-((phase_test - 0.30) / 0.08) ** 2)
metric_test = phase_metrics(phase_test, wave_test)
assert abs(metric_test["extremum_phase_rr"] - 0.30) <= 0.005
assert metric_test["onset_phase_rr_by_fraction"]["0.1"] < metric_test["extremum_phase_rr"]
print("33.05 synthetic_self_test: passed")

In [ ]:
if not REAL_MODE:
    print("33.05 real_data_status: blocked_until_33.03_phase_artifact_exists")
else:
    config_value = os.environ.get("KALMYKOV_EXP02_CONFIG")
    if not config_value:
        raise RuntimeError("Задайте KALMYKOV_EXP02_CONFIG")
    config = json.loads(Path(config_value).expanduser().resolve().read_text(encoding="utf-8"))
    derived_root = Path(config["derived_root"]).expanduser().resolve()
    exploratory_dir = derived_root / "exp02" / "exploratory"
    pulse_path = exploratory_dir / "33.03_pulse_ensembles.provisional.json"
    ensembles = json.loads(pulse_path.read_text(encoding="utf-8"))
    if (
        ensembles.get("schema_version") != 2
        or ensembles.get("status") != "exploratory_hypothesis_not_validated"
    ):
        raise RuntimeError("Неподходящий фазовый артефакт 33.03")
    if ensembles.get("algorithm", {}).get("phase_grid_point_count") != 200:
        raise RuntimeError("33.03 не содержит ожидаемую фазовую сетку")
    results = []
    for item in ensembles["ensembles"]:
        n_cycles = item.get("primary", {}).get("n_cycles")
        if not isinstance(n_cycles, int) or n_cycles < 1:
            raise RuntimeError("33.03 primary.n_cycles должен быть положительным целым числом")
        metric = phase_metrics(item["phase_rr"], item["primary"]["median_mohm"])
        results.append({
            "record_id": item["record_id"],
            "subject_id": item["subject_id"],
            "size_mm": item["size_mm"],
            "mode": item["mode"],
            "signal_column": item["signal_column"],
            "signal_unit": item["signal_unit"],
            "n_cycles": n_cycles,
            **metric,
        })
    artifact = {
        "schema_version": 2,
        "analysis": "33.05_phase_metrics",
        "status": "exploratory_hypothesis_not_validated",
        "analysis_scope": "descriptive_phase_metrics",
        "result_type": "descriptive_phase_metrics",
        "time_axis": "phase_rr",
        "phase_definition": "accepted_R_to_next_accepted_R",
        "onset_definition": "first_threshold_crossing_from_phase_window_start_on_median_centered_ensemble",
        "signal_semantics": ensembles.get("signal_semantics", {}),
        "upstream": {
            "artifact": pulse_path.name,
            "sha256": sha256_file(pulse_path),
        },
        "prohibited_interpretations_without_external_reference": [
            "valve_event", "mechanical_systole", "regional_source_localization",
        ],
        "results": results,
    }
    out_path = exploratory_dir / "33.05_phase_metrics.provisional.json"
    out_path.write_text(
        json.dumps(artifact, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    print("33.05 real_data_status: descriptive_phase_result_written", out_path)

## Граница результата

R-зубец задаёт начало фазовой координаты каждого цикла, но нормированная
фаза не является временем после R и не содержит измеренной групповой задержки.
Поэтому из этого артефакта нельзя получить задержку в секундах или назвать
точку событием клапана, механической систолой или пиком кровенаполнения.
Для физиологического названия и абсолютной временной точки требуется
синхронная эхокардиография, фонокардиография или другой заранее выбранный
метод.